# LayoutLMv3 with Auxiliary Boundary Loss (FUNSD & CORD — Base & Large)

This notebook sets up the exact baseline LayoutLMv3 environment on an **NVIDIA A100 GPU** in Google Colab, clones [pn-minh/unilm](https://github.com/pn-minh/unilm/tree/master/layoutlmv3), trains and evaluates **LayoutLMv3 (Base & Large)** with and without **Auxiliary Boundary Loss** on **FUNSD** and **CORD**, and syncs all metrics and logs directly to **Google Drive**.

## 1. Verify A100 GPU & Mount Google Drive

In [ ]:
!nvidia-smi

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/thesis/layoutlmv3'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('Google Drive output directory:', DRIVE_OUT)
except Exception as e:
    print('Note: google.colab.drive.mount not available in this IDE frontend (will use local /content/runs or rclone):', e)

## 2. Clone Fork, Sync Boundary Loss Files & Bootstrap Environment
Accepts Anaconda Terms of Service and runs scripts/setup_env.sh to create the Python 3.7 conda environment with 	orch==1.10.0+cu111, 	orchvision==0.11.1+cu111, 	ransformers==4.12.5, datasets==1.15.1, 	imm==0.4.12, detectron2, and installs layoutlmft in editable mode.

In [ ]:
import base64, json, os, subprocess

# 1. Clone fork if not already present
if not os.path.exists('/content/unilm/.git'):
    subprocess.run(['git', 'clone', '--filter=blob:none', '--sparse', 'https://github.com/pn-minh/unilm.git', '/content/unilm'], check=True)
    subprocess.run(['git', '-C', '/content/unilm', 'sparse-checkout', 'set', 'layoutlmv3'], check=True)
else:
    subprocess.run(['git', '-C', '/content/unilm', 'pull', '--ff-only'], check=False)

# 2. Sync latest Boundary Loss implementation & scripts into /content/unilm/layoutlmv3
FILES_B64 = json.loads(r'''{"examples/run_xfund.py":"...","layoutlmft/data/cord.py":"JycnClJlZmVyZW5jZTogaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9kYXRhc2V0cy9waWVycmVzaS9jb3JkL2Jsb2IvbWFpbi9jb3JkLnB5CicnJwoKCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGRhdGFzZXRzCmZyb20gbGF5b3V0bG1mdC5kYXRhLmltYWdlX3V0aWxzIGltcG9ydCBsb2FkX2ltYWdlLCBub3JtYWxpemVfYmJveApsb2dnZXIgPSBkYXRhc2V0cy5sb2dnaW5nLmdldF9sb2dnZXIoX19uYW1lX18pCl9DSVRBVElPTiA9ICIiIlwKQGFydGljbGV7cGFyazIwMTljb3JkLAogIHRpdGxlPXtDT1JEOiBBIENvbnNvbGlkYXRlZCBSZWNlaXB0IERhdGFzZXQgZm9yIFBvc3QtT0NSIFBhcnNpbmd9LAogIGF1dGhvcj17UGFyaywgU2V1bmdoeXVuIGFuZCBTaGluLCBTZXVuZyBhbmQgTGVlLCBCYWRvIGFuZCBMZWUsIEp1bnllb3AgYW5kIFN1cmgsIEphZWhldW5nIGFuZCBTZW8sIE1pbmpvb24gYW5kIExlZSwgSHdhbHN1a30KICBib29rdGl0bGU9e0RvY3VtZW50IEludGVsbGlnZW5jZSBXb3Jrc2hvcCBhdCBOZXVyYWwgSW5mb3JtYXRpb24gUHJvY2Vzc2luZyBTeXN0ZW1zfQogIHllYXI9ezIwMTl9Cn0KIiIiCl9ERVNDUklQVElPTiA9ICIiIlwKaHR0cHM6Ly9naXRodWIuY29tL2Nsb3ZhYWkvY29yZC8KIiIiCgpkZWYgcXVhZF90b19ib3gocXVhZCk6CiAgICAjIHRlc3QgODcgaXMgd3JvbmdseSBhbm5vdGF0ZWQKICAgIGJveCA9ICgKICAgICAgICBtYXgoMCwgcXVhZFsieDEiXSksCiAgICAgICAgbWF4KDAsIHF1YWRbInkxIl0pLAogICAgICAgIHF1YWRbIngzIl0sCiAgICAgICAgcXVhZFsieTMiXQogICAgKQogICAgaWYgYm94WzNdIDwgYm94WzFdOgogICAgICAgIGJib3ggPSBsaXN0KGJveCkKICAgICAgICB0bXAgPSBiYm94WzNdCiAgICAgICAgYmJveFszXSA9IGJib3hbMV0KICAgICAgICBiYm94WzFdID0gdG1wCiAgICAgICAgYm94ID0gdHVwbGUoYmJveCkKICAgIGlmIGJveFsyXSA8IGJveFswXToKICAgICAgICBiYm94ID0gbGlzdChib3gpCiAgICAgICAgdG1wID0gYmJveFsyXQogICAgICAgIGJib3hbMl0gPSBiYm94WzBdCiAgICAgICAgYmJveFswXSA9IHRtcAogICAgICAgIGJveCA9IHR1cGxlKGJib3gpCiAgICByZXR1cm4gYm94CgpkZWYgX2dldF9kcml2ZV91cmwodXJsKToKICAgIHNwbGl0X3VybCA9IHVybC5zcGxpdCgnLycpCiAgICByZXR1cm4gZiJodHRwczovL2RyaXZlLnVzZXJjb250ZW50Lmdvb2dsZS5jb20vZG93bmxvYWQ/aWQ9e3NwbGl0X3VybFs1XX0mY29uZmlybT10IgoKX1VSTFMgPSBbCiAgICBfZ2V0X2RyaXZlX3VybCgiaHR0cHM6Ly9kcml2ZS5nb29nbGUuY29tL2ZpbGUvZC8xTXFoVGJjai1BSFhPcVlvZW9oMTJhUlV3SXByelRKWUkvIiksCiAgICBfZ2V0X2RyaXZlX3VybCgiaHR0cHM6Ly9kcml2ZS5nb29nbGUuY29tL2ZpbGUvZC8xd1lkcDVuQzlMbkhRWjJGY21Pb0MwZUNseVd2Y3VBUlUvIikKICAgICMgSWYgeW91IGZhaWxlZCB0byBkb3dubG9hZCB0aGUgZGF0YXNldCB0aHJvdWdoIHRoZSBhdXRvbWF0aWMgZG93bmxvYWRlciwKICAgICMgeW91IGNhbiBkb3dubG9hZCBpdCBtYW51YWxseSBhbmQgbW9kaWZ5IHRoZSBjb2RlIHRvIGdldCB0aGUgbG9jYWwgZGF0YXNldC4KICAgICMgT3IgeW91IGNhbiB1c2UgdGhlIGZvbGxvd2luZyBsaW5rcy4gUGxlYXNlIGZvbGxvdyB0aGUgb3JpZ2luYWwgTElDRU5TRSBvZiBDT1JEIGZvciB1c2FnZS4KICAgICMgImh0dHBzOi8vbGF5b3V0bG0uYmxvYi5jb3JlLndpbmRvd3MubmV0L2NvcmQvQ09SRC0xay0wMDEuemlwIiwKICAgICMgImh0dHBzOi8vbGF5b3V0bG0uYmxvYi5jb3JlLndpbmRvd3MubmV0L2NvcmQvQ09SRC0xay0wMDIuemlwIgpdCgpjbGFzcyBDb3JkQ29uZmlnKGRhdGFzZXRzLkJ1aWxkZXJDb25maWcpOgogICAgIiIiQnVpbGRlckNvbmZpZyBmb3IgQ09SRCIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsICoqa3dhcmdzKToKICAgICAgICAiIiJCdWlsZGVyQ29uZmlnIGZvciBDT1JELgogICAgICAgIEFyZ3M6CiAgICAgICAgICAqKmt3YXJnczoga2V5d29yZCBhcmd1bWVudHMgZm9yd2FyZGVkIHRvIHN1cGVyLgogICAgICAgICIiIgogICAgICAgIHN1cGVyKENvcmRDb25maWcsIHNlbGYpLl9faW5pdF9fKCoqa3dhcmdzKQoKY2xhc3MgQ29yZChkYXRhc2V0cy5HZW5lcmF0b3JCYXNlZEJ1aWxkZXIpOgogICAgQlVJTERFUl9DT05GSUdTID0gWwogICAgICAgIENvcmRDb25maWcobmFtZT0iY29yZCIsIHZlcnNpb249ZGF0YXNldHMuVmVyc2lvbigiMS4wLjAiKSwgZGVzY3JpcHRpb249IkNPUkQgZGF0YXNldCIpLAogICAgXQoKICAgIGRlZiBfaW5mbyhzZWxmKToKICAgICAgICByZXR1cm4gZGF0YXNldHMuRGF0YXNldEluZm8oCiAgICAgICAgICAgIGRlc2NyaXB0aW9uPV9ERVNDUklQVElPTiwKICAgICAgICAgICAgZmVhdHVyZXM9ZGF0YXNldHMuRmVhdHVyZXMoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImlkIjogZGF0YXNldHMuVmFsdWUoInN0cmluZyIpLAogICAgICAgICAgICAgICAgICAgICJ3b3JkcyI6IGRhdGFzZXRzLlNlcXVlbmNlKGRhdGFzZXRzLlZhbHVlKCJzdHJpbmciKSksCiAgICAgICAgICAgICAgICAgICAgImJib3hlcyI6IGRhdGFzZXRzLlNlcXVlbmNlKGRhdGFzZXRzLlNlcXVlbmNlKGRhdGFzZXRzLlZhbHVlKCJpbnQ2NCIpKSksCiAgICAgICAgICAgICAgICAgICAgIm5lcl90YWdzIjogZGF0YXNldHMuU2VxdWVuY2UoCiAgICAgICAgICAgICAgICAgICAgICAgIGRhdGFzZXRzLmZlYXR1cmVzLkNsYXNzTGFiZWwoCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBuYW1lcz1bIk8iLCJCLU1FTlUuTk0iLCJCLU1FTlUuTlVNIiwiQi1NRU5VLlVOSVRQUklDRSIsIkItTUVOVS5DTlQiLCJCLU1FTlUuRElTQ09VTlRQUklDRSIsIkItTUVOVS5QUklDRSIsIkItTUVOVS5JVEVNU1VCVE9UQUwiLCJCLU1FTlUuVkFUWU4iLCJCLU1FTlUuRVRDIiwiQi1NRU5VLlNVQl9OTSIsIkItTUVOVS5TVUJfVU5JVFBSSUNFIiwiQi1NRU5VLlNVQl9DTlQiLCJCLU1FTlUuU1VCX1BSSUNFIiwiQi1NRU5VLlNVQl9FVEMiLCJCLVZPSURfTUVOVS5OTSIsIkItVk9JRF9NRU5VLlBSSUNFIiwiQi1TVUJfVE9UQUwuU1VCVE9UQUxfUFJJQ0UiLCJCLVNVQl9UT1RBTC5ESVNDT1VOVF9QUklDRSIsIkItU1VCX1RPVEFMLlNFUlZJQ0VfUFJJQ0UiLCJCLVNVQl9UT1RBTC5PVEhFUlNWQ19QUklDRSIsIkItU1VCX1RPVEFMLlRBWF9QUklDRSIsIkItU1VCX1RPVEFMLkVUQyIsIkItVE9UQUwuVE9UQUxfUFJJQ0UiLCJCLVRPVEFMLlRPVEFMX0VUQyIsIkItVE9UQUwuQ0FTSFBSSUNFIiwiQi1UT1RBTC5DSEFOR0VQUklDRSIsIkItVE9UQUwuQ1JFRElUQ0FSRFBSSUNFIiwiQi1UT1RBTC5FTU9ORVlQUklDRSIsIkItVE9UQUwuTUVOVVRZUEVfQ05UIiwiQi1UT1RBTC5NRU5VUVRZX0NOVCIsIkktTUVOVS5OTSIsIkktTUVOVS5OVU0iLCJJLU1FTlUuVU5JVFBSSUNFIiwiSS1NRU5VLkNOVCIsIkktTUVOVS5ESVNDT1VOVFBSSUNFIiwiSS1NRU5VLlBSSUNFIiwiSS1NRU5VLklURU1TVUJUT1RBTCIsIkktTUVOVS5WQVRZTiIsIkktTUVOVS5FVEMiLCJJLU1FTlUuU1VCX05NIiwiSS1NRU5VLlNVQl9VTklUUFJJQ0UiLCJJLU1FTlUuU1VCX0NOVCIsIkktTUVOVS5TVUJfUFJJQ0UiLCJJLU1FTlUuU1VCX0VUQyIsIkktVk9JRF9NRU5VLk5NIiwiSS1WT0lEX01FTlUuUFJJQ0UiLCJJLVNVQl9UT1RBTC5TVUJUT1RBTF9QUklDRSIsIkktU1VCX1RPVEFMLkRJU0NPVU5UX1BSSUNFIiwiSS1TVUJfVE9UQUwuU0VSVklDRV9QUklDRSIsIkktU1VCX1RPVEFMLk9USEVSU1ZDX1BSSUNFIiwiSS1TVUJfVE9UQUwuVEFYX1BSSUNFIiwiSS1TVUJfVE9UQUwuRVRDIiwiSS1UT1RBTC5UT1RBTF9QUklDRSIsIkktVE9UQUwuVE9UQUxfRVRDIiwiSS1UT1RBTC5DQVNIUFJJQ0UiLCJJLVRPVEFMLkNIQU5HRVBSSUNFIiwiSS1UT1RBTC5DUkVESVRDQVJEUFJJQ0UiLCJJLVRPVEFMLkVNT05FWVBSSUNFIiwiSS1UT1RBTC5NRU5VVFlQRV9DTlQiLCJJLVRPVEFMLk1FTlVRVFlfQ05UIl0KICAgICAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgICAgICksCiAgICAgICAgICAgICAgICAgICAgImltYWdlIjogZGF0YXNldHMuQXJyYXkzRChzaGFwZT0oMywgMjI0LCAyMjQpLCBkdHlwZT0idWludDgiKSwKICAgICAgICAgICAgICAgICAgICAiaW1hZ2VfcGF0aCI6IGRhdGFzZXRzLlZhbHVlKCJzdHJpbmciKSwKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgKSwKICAgICAgICAgICAgc3VwZXJ2aXNlZF9rZXlzPU5vbmUsCiAgICAgICAgICAgIGNpdGF0aW9uPV9DSVRBVElPTiwKICAgICAgICAgICAgaG9tZXBhZ2U9Imh0dHBzOi8vZ2l0aHViLmNvbS9jbG92YWFpL2NvcmQvIiwKICAgICAgICApCgogICAgZGVmIF9zcGxpdF9nZW5lcmF0b3JzKHNlbGYsIGRsX21hbmFnZXIpOgogICAgICAgICIiIlJldHVybnMgU3BsaXRHZW5lcmF0b3JzLiIiIgogICAgICAgICIiIlVzZXMgbG9jYWwgZmlsZXMgbG9jYXRlZCB3aXRoIGRhdGFfZGlyIiIiCiAgICAgICAgZG93bmxvYWRlZF9maWxlID0gZGxfbWFuYWdlci5kb3dubG9hZF9hbmRfZXh0cmFjdChfVVJMUykKICAgICAgICAjIG1vdmUgZmlsZXMgZnJvbSB0aGUgc2Vjb25kIFVSTCB0b2dldGhlciB3aXRoIGZpbGVzIGZyb20gdGhlIGZpcnN0IG9uZS4KICAgICAgICBkZXN0ID0gUGF0aChkb3dubG9hZGVkX2ZpbGVbMF0pLyJDT1JEIgogICAgICAgIGZvciBzcGxpdCBpbiBbInRyYWluIiwgImRldiIsICJ0ZXN0Il06CiAgICAgICAgICAgIGZvciBmaWxlX3R5cGUgaW4gWyJpbWFnZSIsICJqc29uIl06CiAgICAgICAgICAgICAgICBpZiBzcGxpdCA9PSAidGVzdCIgYW5kIGZpbGVfdHlwZSA9PSAianNvbiI6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHNyY19kaXIgPSBQYXRoKGRvd25sb2FkZWRfZmlsZVsxXSkgLyAiQ09SRCIgLyBzcGxpdCAvIGZpbGVfdHlwZQogICAgICAgICAgICAgICAgZHN0X2RpciA9IGRlc3QgLyBzcGxpdCAvIGZpbGVfdHlwZQogICAgICAgICAgICAgICAgaWYgc3JjX2Rpci5leGlzdHMoKToKICAgICAgICAgICAgICAgICAgICBkc3RfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgICAgICAgICBmb3IgZiBpbiBzcmNfZGlyLml0ZXJkaXIoKToKICAgICAgICAgICAgICAgICAgICAgICAgdGFyZ2V0ID0gZHN0X2RpciAvIGYubmFtZQogICAgICAgICAgICAgICAgICAgICAgICBpZiBub3QgdGFyZ2V0LmV4aXN0cygpOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgb3MucmVuYW1lKGYsIHRhcmdldCkKICAgICAgICByZXR1cm4gWwogICAgICAgICAgICBkYXRhc2V0cy5TcGxpdEdlbmVyYXRvcigKICAgICAgICAgICAgICAgIG5hbWU9ZGF0YXNldHMuU3BsaXQuVFJBSU4sIGdlbl9rd2FyZ3M9eyJmaWxlcGF0aCI6IGRlc3QvInRyYWluIn0KICAgICAgICAgICAgKSwKICAgICAgICAgICAgZGF0YXNldHMuU3BsaXRHZW5lcmF0b3IoCiAgICAgICAgICAgICAgICBuYW1lPWRhdGFzZXRzLlNwbGl0LlZBTElEQVRJT04sIGdlbl9rd2FyZ3M9eyJmaWxlcGF0aCI6IGRlc3QvImRldiJ9CiAgICAgICAgICAgICksCiAgICAgICAgICAgIGRhdGFzZXRzLlNwbGl0R2VuZXJhdG9yKAogICAgICAgICAgICAgICAgbmFtZT1kYXRhc2V0cy5TcGxpdC5URVNULCBnZW5fa3dhcmdzPXsiZmlsZXBhdGgiOiBkZXN0LyJ0ZXN0In0KICAgICAgICAgICAgKSwKICAgICAgICBdCgogICAgZGVmIGdldF9saW5lX2Jib3goc2VsZiwgYmJveHMpOgogICAgICAgIHggPSBbYmJveHNbaV1bal0gZm9yIGkgaW4gcmFuZ2UobGVuKGJib3hzKSkgZm9yIGogaW4gcmFuZ2UoMCwgbGVuKGJib3hzW2ldKSwgMildCiAgICAgICAgeSA9IFtiYm94c1tpXVtqXSBmb3IgaSBpbiByYW5nZShsZW4oYmJveHMpKSBmb3IgaiBpbiByYW5nZSgxLCBsZW4oYmJveHNbaV0pLCAyKV0KCiAgICAgICAgeDAsIHkwLCB4MSwgeTEgPSBtaW4oeCksIG1pbih5KSwgbWF4KHgpLCBtYXgoeSkKCiAgICAgICAgYXNzZXJ0IHgxID49IHgwIGFuZCB5MSA+PSB5MAogICAgICAgIGJib3ggPSBbW3gwLCB5MCwgeDEsIHkxXSBmb3IgXyBpbiByYW5nZShsZW4oYmJveHMpKV0KICAgICAgICByZXR1cm4gYmJveAoKICAgIGRlZiBfZ2VuZXJhdGVfZXhhbXBsZXMoc2VsZiwgZmlsZXBhdGgpOgogICAgICAgIGxvZ2dlci5pbmZvKCLij7MgR2VuZXJhdGluZyBleGFtcGxlcyBmcm9tID0gJXMiLCBmaWxlcGF0aCkKICAgICAgICBhbm5fZGlyID0gb3MucGF0aC5qb2luKGZpbGVwYXRoLCAianNvbiIpCiAgICAgICAgaW1nX2RpciA9IG9zLnBhdGguam9pbihmaWxlcGF0aCwgImltYWdlIikKICAgICAgICBmb3IgZ3VpZCwgZmlsZSBpbiBlbnVtZXJhdGUoc29ydGVkKG9zLmxpc3RkaXIoYW5uX2RpcikpKToKICAgICAgICAgICAgd29yZHMgPSBbXQogICAgICAgICAgICBiYm94ZXMgPSBbXQogICAgICAgICAgICBuZXJfdGFncyA9IFtdCiAgICAgICAgICAgIGZpbGVfcGF0aCA9IG9zLnBhdGguam9pbihhbm5fZGlyLCBmaWxlKQogICAgICAgICAgICB3aXRoIG9wZW4oZmlsZV9wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGY4IikgYXMgZjoKICAgICAgICAgICAgICAgIGRhdGEgPSBqc29uLmxvYWQoZikKICAgICAgICAgICAgaW1hZ2VfcGF0aCA9IG9zLnBhdGguam9pbihpbWdfZGlyLCBmaWxlKQogICAgICAgICAgICBpbWFnZV9wYXRoID0gaW1hZ2VfcGF0aC5yZXBsYWNlKCJqc29uIiwgInBuZyIpCiAgICAgICAgICAgIGltYWdlLCBzaXplID0gbG9hZF9pbWFnZShpbWFnZV9wYXRoKQogICAgICAgICAgICBmb3IgaXRlbSBpbiBkYXRhWyJ2YWxpZF9saW5lIl06CiAgICAgICAgICAgICAgICBjdXJfbGluZV9iYm94ZXMgPSBbXQogICAgICAgICAgICAgICAgbGluZV93b3JkcywgbGFiZWwgPSBpdGVtWyJ3b3JkcyJdLCBpdGVtWyJjYXRlZ29yeSJdCiAgICAgICAgICAgICAgICBsaW5lX3dvcmRzID0gW3cgZm9yIHcgaW4gbGluZV93b3JkcyBpZiB3WyJ0ZXh0Il0uc3RyaXAoKSAhPSAiIl0KICAgICAgICAgICAgICAgIGlmIGxlbihsaW5lX3dvcmRzKSA9PSAwOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICBpZiBsYWJlbCA9PSAib3RoZXIiOgogICAgICAgICAgICAgICAgICAgIGZvciB3IGluIGxpbmVfd29yZHM6CiAgICAgICAgICAgICAgICAgICAgICAgIHdvcmRzLmFwcGVuZCh3WyJ0ZXh0Il0pCiAgICAgICAgICAgICAgICAgICAgICAgIG5lcl90YWdzLmFwcGVuZCgiTyIpCiAgICAgICAgICAgICAgICAgICAgICAgIGN1cl9saW5lX2Jib3hlcy5hcHBlbmQobm9ybWFsaXplX2Jib3gocXVhZF90b19ib3god1sicXVhZCJdKSwgc2l6ZSkpCiAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgIHdvcmRzLmFwcGVuZChsaW5lX3dvcmRzWzBdWyJ0ZXh0Il0pCiAgICAgICAgICAgICAgICAgICAgbmVyX3RhZ3MuYXBwZW5kKCJCLSIgKyBsYWJlbC51cHBlcigpKQogICAgICAgICAgICAgICAgICAgIGN1cl9saW5lX2Jib3hlcy5hcHBlbmQobm9ybWFsaXplX2Jib3gocXVhZF90b19ib3gobGluZV93b3Jkc1swXVsicXVhZCJdKSwgc2l6ZSkpCiAgICAgICAgICAgICAgICAgICAgZm9yIHcgaW4gbGluZV93b3Jkc1sxOl06CiAgICAgICAgICAgICAgICAgICAgICAgIHdvcmRzLmFwcGVuZCh3WyJ0ZXh0Il0pCiAgICAgICAgICAgICAgICAgICAgICAgIG5lcl90YWdzLmFwcGVuZCgiSS0iICsgbGFiZWwudXBwZXIoKSkKICAgICAgICAgICAgICAgICAgICAgICAgY3VyX2xpbmVfYmJveGVzLmFwcGVuZChub3JtYWxpemVfYmJveChxdWFkX3RvX2JveCh3WyJxdWFkIl0pLCBzaXplKSkKICAgICAgICAgICAgICAgICMgYnkgZGVmYXVsdDogLS1zZWdtZW50X2xldmVsX2xheW91dCAxCiAgICAgICAgICAgICAgICAjIGlmIGRvIG5vdCB3YW50IHRvIHVzZSBzZWdtZW50X2xldmVsX2xheW91dCwgY29tbWVudCB0aGUgZm9sbG93aW5nIGxpbmUKICAgICAgICAgICAgICAgIGN1cl9saW5lX2Jib3hlcyA9IHNlbGYuZ2V0X2xpbmVfYmJveChjdXJfbGluZV9iYm94ZXMpCiAgICAgICAgICAgICAgICBiYm94ZXMuZXh0ZW5kKGN1cl9saW5lX2Jib3hlcykKICAgICAgICAgICAgIyB5aWVsZCBndWlkLCB7ImlkIjogc3RyKGd1aWQpLCAid29yZHMiOiB3b3JkcywgImJib3hlcyI6IGJib3hlcywgIm5lcl90YWdzIjogbmVyX3RhZ3MsICJpbWFnZSI6IGltYWdlfQogICAgICAgICAgICB5aWVsZCBndWlkLCB7ImlkIjogc3RyKGd1aWQpLCAid29yZHMiOiB3b3JkcywgImJib3hlcyI6IGJib3hlcywgIm5lcl90YWdzIjogbmVyX3RhZ3MsCiAgICAgICAgICAgICAgICAgICAgICAgICAiaW1hZ2UiOiBpbWFnZSwgImltYWdlX3BhdGgiOiBpbWFnZV9wYXRofQo=","scripts/sync_to_drive.sh":"IyEvdXNyL2Jpbi9lbnYgYmFzaAojIFVwbG9hZCBhIHJ1biBkaXJlY3RvcnkgdG8gR29vZ2xlIERyaXZlIHZpYSBlaXRoZXIgYSBtb3VudGVkIENvbGFiIERyaXZlIGRpcmVjdG9yeSAoRFJJVkVfTU9VTlQpCiMgb3IgcmNsb25lIChEUklWRV9SRU1PVEUpLiBVc2VzIGBjcCAtcmAgLyBgcmNsb25lIGNvcHlgIChuZXZlciBkZWxldGVzIGFueXRoaW5nIG9uIERyaXZlKS4KIwojICAgRFJJVkVfTU9VTlQ9L2NvbnRlbnQvZHJpdmUvTXlEcml2ZSBiYXNoIHNjcmlwdHMvc3luY190b19kcml2ZS5zaCB+L3J1bnMvZnVuc2QtYmFzZS1zZWVkNDIKIyAgIERSSVZFX1JFTU9URT1nZHJpdmUgYmFzaCBzY3JpcHRzL3N5bmNfdG9fZHJpdmUuc2ggfi9ydW5zL2Z1bnNkLWJhc2Utc2VlZDQyCiMKIyBMYXlvdXQgb24gRHJpdmU6IDxEUklWRV9ESVI+LzxydW4gZGlyIG5hbWU+Ly4uLiAgIChEUklWRV9ESVIgZGVmYXVsdHMgdG8gdGhlc2lzL2xheW91dGxtdjMpCnNldCAtZXVvIHBpcGVmYWlsCgpTUkM9IiR7MTo/dXNhZ2U6ICQwIDxsb2NhbF9ydW5fZGlyPn0iCkRSSVZFX1NVQkRJUj0iJHtEUklWRV9ESVI6LXRoZXNpcy9sYXlvdXRsbXYzfS8kKGJhc2VuYW1lICIkU1JDIikiCgppZiBbIC1uICIke0RSSVZFX01PVU5UOi19IiBdOyB0aGVuCiAgWyAtZCAiJERSSVZFX01PVU5UIiBdIHx8IHsgZWNobyAiTW91bnRlZCBEcml2ZSBkaXJlY3RvcnkgJyREUklWRV9NT1VOVCcgbm90IGZvdW5kIjsgZXhpdCAxOyB9CiAgVEFSR0VUX0RJUj0iJERSSVZFX01PVU5ULyREUklWRV9TVUJESVIiCiAgbWtkaXIgLXAgIiRUQVJHRVRfRElSIgogIGNwIC1yICIkU1JDIi8uICIkVEFSR0VUX0RJUiIvCiAgZWNobyAiQ29waWVkIHRvIG1vdW50ZWQgR29vZ2xlIERyaXZlOiAkVEFSR0VUX0RJUiIKICBleGl0IDAKZmkKClJFTU9URT0iJHtEUklWRV9SRU1PVEU6LWdkcml2ZX0iCkRFU1Q9IiREUklWRV9TVUJESVIiCgpjb21tYW5kIC12IHJjbG9uZSA+L2Rldi9udWxsIHx8IHsgZWNobyAicmNsb25lIG5vdCBpbnN0YWxsZWQiOyBleGl0IDE7IH0KcmNsb25lIGxzZCAiJFJFTU9URToiID4vZGV2L251bGwgMj4mMSB8fCB7IGVjaG8gInJjbG9uZSByZW1vdGUgJyRSRU1PVEUnIG5vdCB3b3JraW5nIChydW46IHJjbG9uZSBjb25maWcpIjsgZXhpdCAxOyB9CgpyY2xvbmUgY29weSAiJFNSQyIgIiRSRU1PVEU6JERFU1QiIFwKICAtLXRyYW5zZmVycyA0IC0tY2hlY2tlcnMgOCAtLWRyaXZlLWNodW5rLXNpemUgMTI4TSAtLXJldHJpZXMgNSAtLWxvdy1sZXZlbC1yZXRyaWVzIDIwIC0tcHJvZ3Jlc3MKcmNsb25lIGNoZWNrICIkU1JDIiAiJFJFTU9URTokREVTVCIgLS1vbmUtd2F5ICYmIGVjaG8gIlVwbG9hZGVkIGFuZCB2ZXJpZmllZDogJFJFTU9URTokREVTVCIKCg==","layoutlmft/models/layoutlmv3/modeling_layoutlmv3.py":"..._layoutlmv3.py":"IyBjb2Rpbmc9dXRmLTgKZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxzLmJlcnQuY29uZmlndXJhdGlvbl9iZXJ0IGltcG9ydCBCZXJ0Q29uZmlnCmZyb20gdHJhbnNmb3JtZXJzLnV0aWxzIGltcG9ydCBsb2dnaW5nCgoKbG9nZ2VyID0gbG9nZ2luZy5nZXRfbG9nZ2VyKF9fbmFtZV9fKQoKTEFZT1VUTE1WM19QUkVUUkFJTkVEX0NPTkZJR19BUkNISVZFX01BUCA9IHsKICAgICJsYXlvdXRsbXYzLWJhc2UiOiAiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9taWNyb3NvZnQvbGF5b3V0bG12My1iYXNlL3Jlc29sdmUvbWFpbi9jb25maWcuanNvbiIsCiAgICAibGF5b3V0bG12My1sYXJnZSI6ICJodHRwczovL2h1Z2dpbmdmYWNlLmNvL21pY3Jvc29mdC9sYXlvdXRsbXYzLWxhcmdlL3Jlc29sdmUvbWFpbi9jb25maWcuanNvbiIsCiAgICAjIFNlZSBhbGwgTGF5b3V0TE12MyBtb2RlbHMgYXQgaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9tb2RlbHM/ZmlsdGVyPWxheW91dGxtdjMKfQoKCmNsYXNzIExheW91dExNdjNDb25maWcoQmVydENvbmZpZyk6CiAgICBtb2RlbF90eXBlID0gImxheW91dGxtdjMiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgcGFkX3Rva2VuX2lkPTEsCiAgICAgICAgYm9zX3Rva2VuX2lkPTAsCiAgICAgICAgZW9zX3Rva2VuX2lkPTIsCiAgICAgICAgbWF4XzJkX3Bvc2l0aW9uX2VtYmVkZGluZ3M9MTAyNCwKICAgICAgICBjb29yZGluYXRlX3NpemU9Tm9uZSwKICAgICAgICBzaGFwZV9zaXplPU5vbmUsCiAgICAgICAgaGFzX3JlbGF0aXZlX2F0dGVudGlvbl9iaWFzPUZhbHNlLAogICAgICAgIHJlbF9wb3NfYmlucz0zMiwKICAgICAgICBtYXhfcmVsX3Bvcz0xMjgsCiAgICAgICAgaGFzX3NwYXRpYWxfYXR0ZW50aW9uX2JpYXM9RmFsc2UsCiAgICAgICAgcmVsXzJkX3Bvc19iaW5zPTY0LAogICAgICAgIG1heF9yZWxfMmRfcG9zPTI1NiwKICAgICAgICB2aXN1YWxfZW1iZWQ9VHJ1ZSwKICAgICAgICBtaW09RmFsc2UsCiAgICAgICAgd3BhX3Rhc2s9RmFsc2UsCiAgICAgICAgZGlzY3JldGVfdmFlX3dlaWdodF9wYXRoPScnLAogICAgICAgIGRpc2NyZXRlX3ZhZV90eXBlPSdkYWxsLWUnLAogICAgICAgIGlucHV0X3NpemU9MjI0LAogICAgICAgIHNlY29uZF9pbnB1dF9zaXplPTExMiwKICAgICAgICBkZXZpY2U9J2N1ZGEnLAogICAgICAgIHVzZV9ib3VuZGFyeV9sb3NzPUZhbHNlLAogICAgICAgIGJvdW5kYXJ5X2xvc3Nfd2VpZ2h0PTAuMiwKICAgICAgICBib3VuZGFyeV9wb3Nfd2VpZ2h0PTUuMCwKICAgICAgICBiX2xhYmVsX2lkcz1Ob25lLAogICAgICAgICoqa3dhcmdzCiAgICApOgogICAgICAgICIiIkNvbnN0cnVjdHMgUm9iZXJ0YUNvbmZpZy4iIiIKICAgICAgICBzdXBlcigpLl9faW5pdF9fKHBhZF90b2tlbl9pZD1wYWRfdG9rZW5faWQsIGJvc190b2tlbl9pZD1ib3NfdG9rZW5faWQsIGVvc190b2tlbl9pZD1lb3NfdG9rZW5faWQsICoqa3dhcmdzKQogICAgICAgIHNlbGYubWF4XzJkX3Bvc2l0aW9uX2VtYmVkZGluZ3MgPSBtYXhfMmRfcG9zaXRpb25fZW1iZWRkaW5ncwogICAgICAgIHNlbGYuY29vcmRpbmF0ZV9zaXplID0gY29vcmRpbmF0ZV9zaXplCiAgICAgICAgc2VsZi5zaGFwZV9zaXplID0gc2hhcGVfc2l6ZQogICAgICAgIHNlbGYuaGFzX3JlbGF0aXZlX2F0dGVudGlvbl9iaWFzID0gaGFzX3JlbGF0aXZlX2F0dGVudGlvbl9iaWFzCiAgICAgICAgc2VsZi5yZWxfcG9zX2JpbnMgPSByZWxfcG9zX2JpbnMKICAgICAgICBzZWxmLm1heF9yZWxfcG9zID0gbWF4X3JlbF9wb3MKICAgICAgICBzZWxmLmhhc19zcGF0aWFsX2F0dGVudGlvbl9iaWFzID0gaGFzX3NwYXRpYWxfYXR0ZW50aW9uX2JpYXMKICAgICAgICBzZWxmLnJlbF8yZF9wb3NfYmlucyA9IHJlbF8yZF9wb3NfYmlucwogICAgICAgIHNlbGYubWF4X3JlbF8yZF9wb3MgPSBtYXhfcmVsXzJkX3BvcwogICAgICAgIHNlbGYudmlzdWFsX2VtYmVkID0gdmlzdWFsX2VtYmVkCiAgICAgICAgc2VsZi5taW0gPSBtaW0KICAgICAgICBzZWxmLndwYV90YXNrID0gd3BhX3Rhc2sKICAgICAgICBzZWxmLmRpc2NyZXRlX3ZhZV93ZWlnaHRfcGF0aCA9IGRpc2NyZXRlX3ZhZV93ZWlnaHRfcGF0aAogICAgICAgIHNlbGYuZGlzY3JldGVfdmFlX3R5cGUgPSBkaXNjcmV0ZV92YWVfdHlwZQogICAgICAgIHNlbGYuaW5wdXRfc2l6ZSA9IGlucHV0X3NpemUKICAgICAgICBzZWxmLnNlY29uZF9pbnB1dF9zaXplID0gc2Vjb25kX2lucHV0X3NpemUKICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZQogICAgICAgIHNlbGYudXNlX2JvdW5kYXJ5X2xvc3MgPSB1c2VfYm91bmRhcnlfbG9zcwogICAgICAgIHNlbGYuYm91bmRhcnlfbG9zc193ZWlnaHQgPSBib3VuZGFyeV9sb3NzX3dlaWdodAogICAgICAgIHNlbGYuYm91bmRhcnlfcG9zX3dlaWdodCA9IGJvdW5kYXJ5X3Bvc193ZWlnaHQKICAgICAgICBzZWxmLmJfbGFiZWxfaWRzID0gYl9sYWJlbF9pZHMK","scripts/run_experiments.sh":"IyEvdXNyL2Jpbi9lbnYgYmFzaAojIFJ1biBiYXNlbGluZSBhbmQgQm91bmRhcnkgTG9zcyBleHBlcmltZW50cyBhY3Jvc3MgRlVOU0QgJiBDT1JEIChiYXNlICYgbGFyZ2UpCiMgYW5kIGNvbXBpbGUgYW4gYWdncmVnYXRlZCBldmFsdWF0aW9uIHN1bW1hcnkgdGFibGUsIHN5bmNpbmcgcmVzdWx0cyB0byBHb29nbGUgRHJpdmUuCiMKIyBVc2FnZSAoZnJvbSBsYXlvdXRsbXYzLyB3aXRoIHRoZSBsYXlvdXRsbXYzIGNvbmRhIGVudiBhY3RpdmUpOgojICAgRFJJVkVfTU9VTlQ9L2NvbnRlbnQvZHJpdmUvTXlEcml2ZSBiYXNoIHNjcmlwdHMvcnVuX2V4cGVyaW1lbnRzLnNoCiMgICBEQVRBU0VUUz0iZnVuc2QgY29yZCIgU0laRVM9ImJhc2UgbGFyZ2UiIFZBUklBTlRTPSJiYXNlbGluZSBib3VuZGFyeSIgYmFzaCBzY3JpcHRzL3J1bl9leHBlcmltZW50cy5zaApzZXQgLWV1byBwaXBlZmFpbAoKREFUQVNFVFM9IiR7REFUQVNFVFM6LWZ1bnNkIGNvcmR9IgpTSVpFUz0iJHtTSVpFUzotYmFzZSBsYXJnZX0iClZBUklBTlRTPSIke1ZBUklBTlRTOi1iYXNlbGluZSBib3VuZGFyeX0iClNFRUQ9IiR7U0VFRDotNDJ9IgpPVVQ9IiR7T1VUOi0kSE9NRS9ydW5zfSIKQk9VTkRBUllfTE9TU19XRUlHSFQ9IiR7Qk9VTkRBUllfTE9TU19XRUlHSFQ6LTAuMn0iCkJPVU5EQVJZX1BPU19XRUlHSFQ9IiR7Qk9VTkRBUllfUE9TX1dFSUdIVDotNS4wfSIKClNDUklQVF9ESVI9IiQoY2QgIiQoZGlybmFtZSAiJDAiKSIgJiYgcHdkKSIKCm1rZGlyIC1wICIkT1VUIgoKZm9yIGRhdGFzZXQgaW4gJERBVEFTRVRTOyBkbwogIGZvciBzaXplIGluICRTSVpFUzsgZG8KICAgIGZvciB2YXJpYW50IGluICRWQVJJQU5UUzsgZG8KICAgICAgZWNobyAiPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0iCiAgICAgIGVjaG8gIlJ1bm5pbmc6IGRhdGFzZXQ9JGRhdGFzZXQgfCBzaXplPSRzaXplIHwgdmFyaWFudD0kdmFyaWFudCB8IHNlZWQ9JFNFRUQiCiAgICAgIGVjaG8gIj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09IgogICAgICBpZiBbICIkdmFyaWFudCIgPSAiYm91bmRhcnkiIF07IHRoZW4KICAgICAgICBEQVRBU0VUPSIkZGF0YXNldCIgU0laRT0iJHNpemUiIFNFRUQ9IiRTRUVEIiBPVVQ9IiRPVVQiIFwKICAgICAgICAgIFVTRV9CT1VOREFSWV9MT1NTPTEgXAogICAgICAgICAgQk9VTkRBUllfTE9TU19XRUlHSFQ9IiRCT1VOREFSWV9MT1NTX1dFSUdIVCIgXAogICAgICAgICAgQk9VTkRBUllfUE9TX1dFSUdIVD0iJEJPVU5EQVJZX1BPU19XRUlHSFQiIFwKICAgICAgICAgIGJhc2ggIiRTQ1JJUFRfRElSL3J1bl9iYXNlbGluZS5zaCIgdHJhaW4KICAgICAgZWxzZQogICAgICAgIERBVEFTRVQ9IiRkYXRhc2V0IiBTSVpFPSIkc2l6ZSIgU0VFRD0iJFNFRUQiIE9VVD0iJE9VVCIgXAogICAgICAgICAgVVNFX0JPVU5EQVJZX0xPU1M9MCBcCiAgICAgICAgICBiYXNoICIkU0NSSVBUX0RJUi9ydW5fYmFzZWxpbmUuc2giIHRyYWluCiAgICAgIGZpCiAgICBkb25lCiAgZG9uZQpkb25lCgojIEFnZ3JlZ2F0ZSBldmFsdWF0aW9uIG1ldHJpY3MgYWNyb3NzIGFsbCBjb21wbGV0ZWQgcnVucyBpbnRvIENTViBhbmQgTWFya2Rvd24gdGFibGVzClNVTU1BUllfRElSPSIkT1VUL3N1bW1hcnkiCm1rZGlyIC1wICIkU1VNTUFSWV9ESVIiCgpweXRob24gLSAiJE9VVCIgIiRTVU1NQVJZX0RJUiIgPDwnRU9GJwppbXBvcnQgY3N2CmltcG9ydCBnbG9iCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCgpvdXRfZGlyLCBzdW1tYXJ5X2RpciA9IHN5cy5hcmd2WzFdLCBzeXMuYXJndlsyXQpyb3dzID0gW10KCmZvciBldmFsX3BhdGggaW4gc29ydGVkKGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4ob3V0X2RpciwgIioiLCAiZXZhbF9yZXN1bHRzLmpzb24iKSkpOgogICAgcnVuX2RpciA9IG9zLnBhdGguZGlybmFtZShldmFsX3BhdGgpCiAgICBydW5fbmFtZSA9IG9zLnBhdGguYmFzZW5hbWUocnVuX2RpcikKICAgIGlmIHJ1bl9uYW1lID09ICJzdW1tYXJ5IjoKICAgICAgICBjb250aW51ZQogICAgd2l0aCBvcGVuKGV2YWxfcGF0aCwgInIiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIG1ldHJpY3MgPSBqc29uLmxvYWQoZikKICAgIGluZm8gPSB7fQogICAgaW5mb19wYXRoID0gb3MucGF0aC5qb2luKHJ1bl9kaXIsICJydW5faW5mby50eHQiKQogICAgaWYgb3MucGF0aC5leGlzdHMoaW5mb19wYXRoKToKICAgICAgICB3aXRoIG9wZW4oaW5mb19wYXRoLCAiciIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgICAgICBmb3IgdG9rZW4gaW4gbGluZS5zdHJpcCgpLnNwbGl0KCk6CiAgICAgICAgICAgICAgICAgICAgaWYgIj0iIGluIHRva2VuOgogICAgICAgICAgICAgICAgICAgICAgICBrLCB2ID0gdG9rZW4uc3BsaXQoIj0iLCAxKQogICAgICAgICAgICAgICAgICAgICAgICBpbmZvW2tdID0gdgogICAgcm93cy5hcHBlbmQoewogICAgICAgICJydW5fbmFtZSI6IHJ1bl9uYW1lLAogICAgICAgICJkYXRhc2V0IjogaW5mby5nZXQoImRhdGFzZXQiLCAiIiksCiAgICAgICAgInNpemUiOiBpbmZvLmdldCgic2l6ZSIsICIiKSwKICAgICAgICAidXNlX2JvdW5kYXJ5X2xvc3MiOiBpbmZvLmdldCgidXNlX2JvdW5kYXJ5X2xvc3MiLCAiMCIpLAogICAgICAgICJib3VuZGFyeV9sb3NzX3dlaWdodCI6IGluZm8uZ2V0KCJib3VuZGFyeV9sb3NzX3dlaWdodCIsICIwLjAiKSwKICAgICAgICAiYm91bmRhcnlfcG9zX3dlaWdodCI6IGluZm8uZ2V0KCJib3VuZGFyeV9wb3Nfd2VpZ2h0IiwgIjEuMCIpLAogICAgICAgICJwcmVjaXNpb24iOiBtZXRyaWNzLmdldCgiZXZhbF9wcmVjaXNpb24iLCAwLjApLAogICAgICAgICJyZWNhbGwiOiBtZXRyaWNzLmdldCgiZXZhbF9yZWNhbGwiLCAwLjApLAogICAgICAgICJmMSI6IG1ldHJpY3MuZ2V0KCJldmFsX2YxIiwgMC4wKSwKICAgICAgICAiYWNjdXJhY3kiOiBtZXRyaWNzLmdldCgiZXZhbF9hY2N1cmFjeSIsIDAuMCksCiAgICAgICAgImxvc3MiOiBtZXRyaWNzLmdldCgiZXZhbF9sb3NzIiwgMC4wKSwKICAgIH0pCgpjc3ZfcGF0aCA9IG9zLnBhdGguam9pbihzdW1tYXJ5X2RpciwgInN1bW1hcnlfbWV0cmljcy5jc3YiKQptZF9wYXRoID0gb3MucGF0aC5qb2luKHN1bW1hcnlfZGlyLCAic3VtbWFyeV9tZXRyaWNzLm1kIikKCmZpZWxkbmFtZXMgPSBbCiAgICAicnVuX25hbWUiLCAiZGF0YXNldCIsICJzaXplIiwgInVzZV9ib3VuZGFyeV9sb3NzIiwKICAgICJib3VuZGFyeV9sb3NzX3dlaWdodCIsICJib3VuZGFyeV9wb3Nfd2VpZ2h0IiwKICAgICJwcmVjaXNpb24iLCAicmVjYWxsIiwgImYxIiwgImFjY3VyYWN5IiwgImxvc3MiCl0Kd2l0aCBvcGVuKGNzdl9wYXRoLCAidyIsIG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICB3cml0ZXIgPSBjc3YuRGljdFdyaXRlcihmLCBmaWVsZG5hbWVzPWZpZWxkbmFtZXMpCiAgICB3cml0ZXIud3JpdGVoZWFkZXIoKQogICAgd3JpdGVyLndyaXRlcm93cyhyb3dzKQoKd2l0aCBvcGVuKG1kX3BhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgIGYud3JpdGUoIiMgTGF5b3V0TE12MyBCb3VuZGFyeSBMb3NzIEV2YWx1YXRpb24gU3VtbWFyeVxuXG4iKQogICAgZi53cml0ZSgifCBSdW4gfCBEYXRhc2V0IHwgU2l6ZSB8IEJvdW5kYXJ5IExvc3MgfCBXZWlnaHQgKM67KSB8IFBvcyBXZWlnaHQgfCBQcmVjaXNpb24gfCBSZWNhbGwgfCBGMSB8IEFjY3VyYWN5IHxcbiIpCiAgICBmLndyaXRlKCJ8LS0tfC0tLXwtLS18LS0tfC0tLTp8LS0tOnwtLS06fC0tLTp8LS0tOnwtLS06fFxuIikKICAgIGZvciByIGluIHJvd3M6CiAgICAgICAgZi53cml0ZSgKICAgICAgICAgICAgZiJ8IGB7clsncnVuX25hbWUnXX1gIHwge3JbJ2RhdGFzZXQnXX0gfCB7clsnc2l6ZSddfSB8IHtyWyd1c2VfYm91bmRhcnlfbG9zcyddfSB8ICIKICAgICAgICAgICAgZiJ7clsnYm91bmRhcnlfbG9zc193ZWlnaHQnXX0gfCB7clsnYm91bmRhcnlfcG9zX3dlaWdodCddfSB8ICIKICAgICAgICAgICAgZiJ7clsncHJlY2lzaW9uJ106LjRmfSB8IHtyWydyZWNhbGwnXTouNGZ9IHwgKip7clsnZjEnXTouNGZ9KiogfCB7clsnYWNjdXJhY3knXTouNGZ9IHxcbiIKICAgICAgICApCgpwcmludChvcGVuKG1kX3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikucmVhZCgpKQpFT0YKCmlmIFsgLW4gIiR7RFJJVkVfUkVNT1RFOi19IiBdIHx8IFsgLW4gIiR7RFJJVkVfTU9VTlQ6LX0iIF07IHRoZW4KICBiYXNoICIkU0NSSVBUX0RJUi9zeW5jX3RvX2RyaXZlLnNoIiAiJFNVTU1BUllfRElSIgpmaQoK","scripts/run_baseline.sh":"IyEvdXNyL2Jpbi9lbnYgYmFzaAojIFRyYWluIC8gZXZhbHVhdGUgTGF5b3V0TE12MyBvbiBGVU5TRCBvciBDT1JEIChiYXNlIG9yIGxhcmdlKSwgb3B0aW9uYWxseSB1cGxvYWRpbmcgcmVzdWx0cyB0byBHb29nbGUgRHJpdmUuCiMKIyAgIGJhc2ggc2NyaXB0cy9ydW5fYmFzZWxpbmUuc2ggZXZhbC1yZWxlYXNlZCAgICAgICAgICAgICAgICAgICAgICAjIEZVTlNEIGJhc2UgcmVsZWFzZWQgY2twdCAoUkVBRE1FIG51bWJlcnMpCiMgICBTSVpFPWxhcmdlIGJhc2ggc2NyaXB0cy9ydW5fYmFzZWxpbmUuc2ggZXZhbC1yZWxlYXNlZCAgICAgICAgICAgIyBGVU5TRCBsYXJnZSByZWxlYXNlZCBja3B0CiMgICBHUFVTPTEgU0VFRD00MiBiYXNoIHNjcmlwdHMvcnVuX2Jhc2VsaW5lLnNoIHRyYWluICAgICAgICAgICAgICAgIyBGVU5TRCBiYXNlLCBSRUFETUUgcmVjaXBlCiMgICBEQVRBU0VUPWNvcmQgTFI9NWUtNSBFRkZfQkFUQ0g9NjQgU1RFUFM9MTAwMCBiYXNoIHNjcmlwdHMvcnVuX2Jhc2VsaW5lLnNoIHRyYWluICAgIyBDT1JEICh2YWx1ZXMgVU5WRVJJRklFRCkKIyAgIFNJWkU9bGFyZ2UgTFI9Li4uIGJhc2ggc2NyaXB0cy9ydW5fYmFzZWxpbmUuc2ggdHJhaW4gICAgICAgICAgICAjIGxhcmdlOiBSRUFETUUgZ2l2ZXMgbm8gcmVjaXBlLCBMUiBtdXN0IGJlIHNldAojCiMgRW52IHZhcnM6IERBVEFTRVQ9ZnVuc2R8Y29yZCAgU0laRT1iYXNlfGxhcmdlICBHUFVTICBTRUVEICBMUiAgU1RFUFMgIEVGRl9CQVRDSCAgUEVSX0RFVklDRSAgT1VUICBQT1JUCiMgICAgICAgICAgIFVTRV9CT1VOREFSWV9MT1NTPTB8MSAgQk9VTkRBUllfTE9TU19XRUlHSFQgKGRlZmF1bHQgMC4yKSAgQk9VTkRBUllfUE9TX1dFSUdIVCAoZGVmYXVsdCA1LjApCiMgICAgICAgICAgIERSSVZFX01PVU5UIChlLmcuIC9jb250ZW50L2RyaXZlL015RHJpdmUpIG9yIERSSVZFX1JFTU9URSAoZS5nLiBnZHJpdmUpCiMgICAgICAgICAgIERSSVZFX0RJUiAoZGVmYXVsdCB0aGVzaXMvbGF5b3V0bG12MykgLT4gdXBsb2FkIHJ1biBkaXIgd2hlbiBmaW5pc2hlZAojCiMgRGVmYXVsdHM6CiMgICAqIEZVTlNEIChiYXNlIC8gbGFyZ2UpOiBsciAxZS01LCAxMDAwIHN0ZXBzLCBlZmZlY3RpdmUgYmF0Y2ggMTYgKD0gOCBHUFVzIHggMiBpbiB0aGUgUkVBRE1FKS4KIyAgICogQ09SRCAgKGJhc2UgLyBsYXJnZSk6IGxyIDVlLTUsIDEwMDAgc3RlcHMsIGVmZmVjdGl2ZSBiYXRjaCA2NC4KIyBFZmZlY3RpdmUgYmF0Y2ggaXMga2VwdCBjb25zdGFudCBmb3IgYW55IEdQVSBjb3VudCB2aWEgZ3JhZGllbnQgYWNjdW11bGF0aW9uLgojIFJ1biBmcm9tIGxheW91dGxtdjMvIHdpdGggdGhlIGNvbmRhIGVudiBhY3RpdmUuCnNldCAtZXVvIHBpcGVmYWlsCgpNT0RFPSIkezE6P3VzYWdlOiAkMCBldmFsLXJlbGVhc2VkfHRyYWlufSIKREFUQVNFVD0iJHtEQVRBU0VUOi1mdW5zZH0iClNJWkU9IiR7U0laRTotYmFzZX0iCkdQVVM9IiR7R1BVUzotMX0iClNFRUQ9IiR7U0VFRDotNDJ9IgpQT1JUPSIke1BPUlQ6LTQzOTh9IgpQRVJfREVWSUNFPSIke1BFUl9ERVZJQ0U6LTJ9IgpPVVQ9IiR7T1VUOi0kSE9NRS9ydW5zfSIKVVNFX0JPVU5EQVJZX0xPU1M9IiR7VVNFX0JPVU5EQVJZX0xPU1M6LTB9IgpCT1VOREFSWV9MT1NTX1dFSUdIVD0iJHtCT1VOREFSWV9MT1NTX1dFSUdIVDotMC4yfSIKQk9VTkRBUllfUE9TX1dFSUdIVD0iJHtCT1VOREFSWV9QT1NfV0VJR0hUOi01LjB9IgpTQVZFX01PREVMPSIke1NBVkVfTU9ERUw6LTB9IgoKaWYgWyAiJERBVEFTRVQiID0gImZ1bnNkIiBdOyB0aGVuCiAgTFI9IiR7TFI6LTFlLTV9IjsgU1RFUFM9IiR7U1RFUFM6LTEwMDB9IjsgRUZGX0JBVENIPSIke0VGRl9CQVRDSDotMTZ9IgplbGlmIFsgIiREQVRBU0VUIiA9ICJjb3JkIiBdOyB0aGVuCiAgTFI9IiR7TFI6LTVlLTV9IjsgU1RFUFM9IiR7U1RFUFM6LTEwMDB9IjsgRUZGX0JBVENIPSIke0VGRl9CQVRDSDotNjR9IgplbHNlCiAgOiAiJHtMUjo/c2V0IExSfSIKICA6ICIke1NURVBTOj9zZXQgU1RFUFN9IgogIDogIiR7RUZGX0JBVENIOj9zZXQgRUZGX0JBVENIfSIKZmkKCkRFTk9NPSQoKFBFUl9ERVZJQ0UgKiBHUFVTKSkKaWYgWyAkKChFRkZfQkFUQ0ggJSBERU5PTSkpIC1uZSAwIF07IHRoZW4gZWNobyAiRUZGX0JBVENIICgkRUZGX0JBVENIKSBtdXN0IGJlIGRpdmlzaWJsZSBieSBQRVJfREVWSUNFKkdQVVMgKCRERU5PTSkiOyBleGl0IDE7IGZpCkFDQ1VNPSQoKEVGRl9CQVRDSCAvIERFTk9NKSkKCkxBVU5DSD0icHl0aG9uIC1tIHRvcmNoLmRpc3RyaWJ1dGVkLmxhdW5jaCAtLW5wcm9jX3Blcl9ub2RlPSRHUFVTIC0tbWFzdGVyX3BvcnQgJFBPUlQiCkNPTU1PTj0iLS1kYXRhc2V0X25hbWUgJERBVEFTRVQgLS1zZWdtZW50X2xldmVsX2xheW91dCAxIC0tdmlzdWFsX2VtYmVkIDEgLS1pbnB1dF9zaXplIDIyNCAtLWRhdGFsb2FkZXJfbnVtX3dvcmtlcnMgOCAtLXNlZWQgJFNFRUQgLS1vdmVyd3JpdGVfb3V0cHV0X2RpciIKCkJPVU5EQVJZX0FSR1M9IiIKUlVOX1RBRz0iYmFzZWxpbmUiCmlmIFsgIiRVU0VfQk9VTkRBUllfTE9TUyIgPSAiMSIgXSB8fCBbICIkVVNFX0JPVU5EQVJZX0xPU1MiID0gInRydWUiIF07IHRoZW4KICBCT1VOREFSWV9BUkdTPSItLXVzZV9ib3VuZGFyeV9sb3NzIFRydWUgLS1ib3VuZGFyeV9sb3NzX3dlaWdodCAkQk9VTkRBUllfTE9TU19XRUlHSFQgLS1ib3VuZGFyeV9wb3Nfd2VpZ2h0ICRCT1VOREFSWV9QT1NfV0VJR0hUIgogIFJVTl9UQUc9ImJvdW5kYXJ5LXcke0JPVU5EQVJZX0xPU1NfV0VJR0hUfS1wdyR7Qk9VTkRBUllfUE9TX1dFSUdIVH0iCmZpCgpSVU5fRElSPSIkT1VULyREQVRBU0VULSRTSVpFLSRSVU5fVEFHLXNlZWQkU0VFRCIKCmNhc2UgIiRNT0RFIiBpbgogIGV2YWwtcmVsZWFzZWQpCiAgICBbICIkREFUQVNFVCIgPSAiZnVuc2QiIF0gfHwgeyBlY2hvICJSRUFETUUgb25seSBsaXN0cyByZWxlYXNlZCBGVU5TRCBjaGVja3BvaW50cyI7IGV4aXQgMTsgfQogICAgUlVOX0RJUj0iJE9VVC9yZWxlYXNlZC0kREFUQVNFVC0kU0laRSIKICAgICRMQVVOQ0ggZXhhbXBsZXMvcnVuX2Z1bnNkX2NvcmQucHkgJENPTU1PTiAtLWRvX2V2YWwgXAogICAgICAtLW1vZGVsX25hbWVfb3JfcGF0aCAiSFlQSlVEWS9sYXlvdXRsbXYzLSRTSVpFLWZpbmV0dW5lZC1mdW5zZCIgXAogICAgICAtLW91dHB1dF9kaXIgIiRSVU5fRElSIgogICAgOzsKICB0cmFpbikKICAgICRMQVVOQ0ggZXhhbXBsZXMvcnVuX2Z1bnNkX2NvcmQucHkgJENPTU1PTiAtLWRvX3RyYWluIC0tZG9fZXZhbCBcCiAgICAgIC0tbW9kZWxfbmFtZV9vcl9wYXRoICJtaWNyb3NvZnQvbGF5b3V0bG12My0kU0laRSIgXAogICAgICAtLW91dHB1dF9kaXIgIiRSVU5fRElSIiBcCiAgICAgIC0tbWF4X3N0ZXBzICIkU1RFUFMiIC0tc2F2ZV9zdGVwcyAtMSAtLWV2YWx1YXRpb25fc3RyYXRlZ3kgc3RlcHMgLS1ldmFsX3N0ZXBzIDEwMCBcCiAgICAgIC0tbGVhcm5pbmdfcmF0ZSAiJExSIiAtLXBlcl9kZXZpY2VfdHJhaW5fYmF0Y2hfc2l6ZSAiJFBFUl9ERVZJQ0UiIC0tZ3JhZGllbnRfYWNjdW11bGF0aW9uX3N0ZXBzICIkQUNDVU0iIFwKICAgICAgJEJPVU5EQVJZX0FSR1MKICAgIGlmIFsgIiRTQVZFX01PREVMIiA9ICIwIiBdOyB0aGVuCiAgICAgIHJtIC1mICIkUlVOX0RJUi9weXRvcmNoX21vZGVsLmJpbiIgIiRSVU5fRElSL29wdGltaXplci5wdCIgIiRSVU5fRElSL3NjaGVkdWxlci5wdCIKICAgIGZpCiAgICA7OwogICopIGVjaG8gInVua25vd24gbW9kZSAkTU9ERSI7IGV4aXQgMTs7CmVzYWMKCmNhdCAiJFJVTl9ESVIvZXZhbF9yZXN1bHRzLmpzb24iCgojIFNhdmUgdGhlIGV4YWN0IHNldHRpbmdzIG5leHQgdG8gdGhlIHJlc3VsdHMgKGZvciB0aGUgdGhlc2lzJyByZXByb2R1Y2liaWxpdHkgYXBwZW5kaXgpCnsKICBlY2hvICJtb2RlPSRNT0RFIGRhdGFzZXQ9JERBVEFTRVQgc2l6ZT0kU0laRSBncHVzPSRHUFVTIHNlZWQ9JFNFRUQgbHI9JHtMUjotbi9hfSBzdGVwcz0ke1NURVBTOi1uL2F9IGVmZl9iYXRjaD0ke0VGRl9CQVRDSDotbi9hfSIKICBlY2hvICJ1c2VfYm91bmRhcnlfbG9zcz0kVVNFX0JPVU5EQVJZX0xPU1MgYm91bmRhcnlfbG9zc193ZWlnaHQ9JEJPVU5EQVJZX0xPU1NfV0VJR0hUIGJvdW5kYXJ5X3Bvc193ZWlnaHQ9JEJPVU5EQVJZX1BPU19XRUlHSFQiCiAgZWNobyAiZ2l0PSQoZ2l0IHJldi1wYXJzZSBIRUFEIDI+L2Rldi9udWxsKSBkaXJ0eT0kKGdpdCBzdGF0dXMgLS1wb3JjZWxhaW4gMj4vZGV2L251bGwgfCB3YyAtbCkiCiAgbnZpZGlhLXNtaSAtLXF1ZXJ5LWdwdT1uYW1lIC0tZm9ybWF0PWNzdixub2hlYWRlciB8IGhlYWQgLTEKfSA+ICIkUlVOX0RJUi9ydW5faW5mby50eHQiCgppZiBbIC1uICIke0RSSVZFX1JFTU9URTotfSIgXSB8fCBbIC1uICIke0RSSVZFX01PVU5UOi19IiBdOyB0aGVuCiAgYmFzaCAiJChkaXJuYW1lICIkMCIpL3N5bmNfdG9fZHJpdmUuc2giICIkUlVOX0RJUiIKZmkKCg==","scripts/setup_env.sh":"IyEvdXNyL2Jpbi9lbnYgYmFzaAojIEJvb3RzdHJhcCB0aGUgTGF5b3V0TE12MyBlbnZpcm9ubWVudCBvbiBhIGZyZXNoIExpbnV4IEdQVSBtYWNoaW5lLCBmb2xsb3dpbmcgbGF5b3V0bG12My9SRUFETUUubWQuCiMKIyBEZXZpYXRpb25zIGZyb20gdGhlIFJFQURNRSAob24gcHVycG9zZSk6CiMgICAqIGNsb25lcyBZT1VSIGZvcmsgKHNldCBSRVBPX1VSTCkgaW5zdGVhZCBvZiBtaWNyb3NvZnQvdW5pbG0KIyAgICogaW5zdGFsbHMgdG9yY2ggQkVGT1JFIHJlcXVpcmVtZW50cy50eHQgKHRpbW0gd291bGQgb3RoZXJ3aXNlIHB1bGwgYSBkaWZmZXJlbnQgdG9yY2ggZmlyc3QpCiMgICAqIHBpbnMgcHJvdG9idWY9PTMuMjAuMyAodGVuc29yYm9hcmQgMi43IC8gZGF0YXNldHMgMS4xNSBicmVhayB3aXRoIHByb3RvYnVmPj00KQojCiMgVGFyZ2V0IEdQVXM6IEFtcGVyZSAoQTEwMCAvIEE2MDAwIC8gUlRYIDMwOTAgLyBBMTAgLi4uKS4gdG9yY2ggMS4xMCtjdTExMSBzaGlwcyBrZXJuZWxzIHVwIHRvIHNtXzg2CiMgYW5kIGhhcyBOTyBuYXRpdmUga2VybmVscyBmb3IgQWRhIChzbV84OSkgb3IgSG9wcGVyIChzbV85MCkuCiMKIyBVc2FnZTogIGJhc2ggc2V0dXBfZW52LnNoICAgICAgICAgICAgKHJ1biBmcm9tIGFueXdoZXJlKQojICAgICAgICAgUkVQT19VUkw9Li4uIFdPUktESVI9L3dvcmtzcGFjZSBiYXNoIHNldHVwX2Vudi5zaApzZXQgLWV1byBwaXBlZmFpbAoKUkVQT19VUkw9IiR7UkVQT19VUkw6LWh0dHBzOi8vZ2l0aHViLmNvbS9wbi1taW5oL3VuaWxtLmdpdH0iCldPUktESVI9IiR7V09SS0RJUjotJEhPTUV9IgpFTlZfTkFNRT0iJHtFTlZfTkFNRTotbGF5b3V0bG12M30iCgojIDEuIGNvbmRhIChpbnN0YWxsIE1pbmljb25kYSBpZiBtaXNzaW5nKQpzZXQgK3UKaWYgISBjb21tYW5kIC12IGNvbmRhID4vZGV2L251bGwgMj4mMTsgdGhlbgogIGlmIFsgISAtZCAiJEhPTUUvbWluaWNvbmRhMyIgXTsgdGhlbgogICAgY3VybCAtZnNTTCBodHRwczovL3JlcG8uYW5hY29uZGEuY29tL21pbmljb25kYS9NaW5pY29uZGEzLWxhdGVzdC1MaW51eC14ODZfNjQuc2ggLW8gL3RtcC9taW5pY29uZGEuc2gKICAgIGJhc2ggL3RtcC9taW5pY29uZGEuc2ggLWIgLXAgIiRIT01FL21pbmljb25kYTMiCiAgZmkKICAjIHNoZWxsY2hlY2sgZGlzYWJsZT1TQzEwOTEKICBzb3VyY2UgIiRIT01FL21pbmljb25kYTMvZXRjL3Byb2ZpbGUuZC9jb25kYS5zaCIKZWxzZQogICMgc2hlbGxjaGVjayBkaXNhYmxlPVNDMTA5MQogIHNvdXJjZSAiJChjb25kYSBpbmZvIC0tYmFzZSkvZXRjL3Byb2ZpbGUuZC9jb25kYS5zaCIKZmkKCiMgQWNjZXB0IEFuYWNvbmRhIFRvUyBvbiBuZXdlciBNaW5pY29uZGEgcmVsZWFzZXMgKGNvbmRhID49IDI1LngpCmNvbmRhIHRvcyBhY2NlcHQgLS1vdmVycmlkZS1jaGFubmVscyAtLWNoYW5uZWwgaHR0cHM6Ly9yZXBvLmFuYWNvbmRhLmNvbS9wa2dzL21haW4gPi9kZXYvbnVsbCAyPiYxIHx8IHRydWUKY29uZGEgdG9zIGFjY2VwdCAtLW92ZXJyaWRlLWNoYW5uZWxzIC0tY2hhbm5lbCBodHRwczovL3JlcG8uYW5hY29uZGEuY29tL3BrZ3MvciA+L2Rldi9udWxsIDI+JjEgfHwgdHJ1ZQoKIyAyLiBlbnYgKFJFQURNRTogcHl0aG9uIDMuNykKaWYgISBjb25kYSBlbnYgbGlzdCB8IGF3ayAne3ByaW50ICQxfScgfCBncmVwIC1xeCAiJEVOVl9OQU1FIjsgdGhlbgogIGNvbmRhIGNyZWF0ZSAteSAtLW5hbWUgIiRFTlZfTkFNRSIgcHl0aG9uPTMuNwpmaQpjb25kYSBhY3RpdmF0ZSAiJEVOVl9OQU1FIgpzZXQgLXUKCiMgMy4gY29kZQpta2RpciAtcCAiJFdPUktESVIiCmlmIFsgISAtZCAiJFdPUktESVIvdW5pbG0vLmdpdCIgXTsgdGhlbgogIGdpdCBjbG9uZSAiJFJFUE9fVVJMIiAiJFdPUktESVIvdW5pbG0iCmZpCmNkICIkV09SS0RJUi91bmlsbS9sYXlvdXRsbXYzIgoKIyA0LiB0b3JjaCAvIHRvcmNodmlzaW9uIChSRUFETUU6IDEuMTAuMCtjdTExMSAvIDAuMTEuMStjdTExMSkKcGlwIGluc3RhbGwgInNldHVwdG9vbHM9PTU5LjUuMCIgIndoZWVsPDAuNDAuMCIKcGlwIGluc3RhbGwgdG9yY2g9PTEuMTAuMCtjdTExMSB0b3JjaHZpc2lvbj09MC4xMS4xK2N1MTExIC1mIGh0dHBzOi8vZG93bmxvYWQucHl0b3JjaC5vcmcvd2hsL3RvcmNoX3N0YWJsZS5odG1sCgojIDUuIHB5dGhvbiBkZXBzCkRTX0JVSUxEX09QUz0wIHBpcCBpbnN0YWxsIC1yIHJlcXVpcmVtZW50cy50eHQKcGlwIGluc3RhbGwgInByb3RvYnVmPT0zLjIwLjMiICJzZXR1cHRvb2xzPT01OS41LjAiCgojIDYuIGRldGVjdHJvbjIgKG9ubHkgbmVlZGVkIGZvciBQdWJMYXlOZXQgLyBvYmplY3RfZGV0ZWN0aW9uOyBwcmVidWlsdCB3aGVlbCBmb3IgY3UxMTEgKyB0b3JjaDEuMTApCnB5dGhvbiAtbSBwaXAgaW5zdGFsbCBkZXRlY3Ryb24yIC1mIGh0dHBzOi8vZGwuZmJhaXB1YmxpY2ZpbGVzLmNvbS9kZXRlY3Ryb24yL3doZWVscy9jdTExMS90b3JjaDEuMTAvaW5kZXguaHRtbAoKIyA3LiB0aGlzIHJlcG8gKHByb3ZpZGVzIHRoZSBgbGF5b3V0bG1mdGAgcGFja2FnZSkKcGlwIGluc3RhbGwgLWUgLgoKIyA4LiBzbW9rZSB0ZXN0CnB5dGhvbiAtIDw8J0VPRicKaW1wb3J0IHRvcmNoLCB0cmFuc2Zvcm1lcnMsIGRhdGFzZXRzLCB0aW1tCnByaW50KCJ0b3JjaCIsIHRvcmNoLl9fdmVyc2lvbl9fLCAifCBjdWRhIiwgdG9yY2gudmVyc2lvbi5jdWRhLCAifCBhdmFpbGFibGUiLCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpKQpwcmludCgidHJhbnNmb3JtZXJzIiwgdHJhbnNmb3JtZXJzLl9fdmVyc2lvbl9fLCAifCBkYXRhc2V0cyIsIGRhdGFzZXRzLl9fdmVyc2lvbl9fLCAifCB0aW1tIiwgdGltbS5fX3ZlcnNpb25fXykKaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgIHByaW50KCJncHUiLCB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgwKSwgInwgY2FwYWJpbGl0eSIsIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApKQogICAgcHJpbnQoImFyY2ggbGlzdCBpbiB0aGlzIHRvcmNoIGJ1aWxkOiIsIHRvcmNoLmN1ZGEuZ2V0X2FyY2hfbGlzdCgpKQogICAgeCA9IHRvcmNoLnJhbmRuKDY0LCA2NCwgZGV2aWNlPSJjdWRhIik7IHByaW50KCJtYXRtdWwgb2s6IiwgKHggQCB4KS5zdW0oKS5pdGVtKCkgPT0gKHggQCB4KS5zdW0oKS5pdGVtKCkpCmltcG9ydCBsYXlvdXRsbWZ0CmZyb20gbGF5b3V0bG1mdC5tb2RlbHMubGF5b3V0bG12MyBpbXBvcnQgTGF5b3V0TE12M0ZvclRva2VuQ2xhc3NpZmljYXRpb24gICMgbm9xYTogRjQwMQppbXBvcnQgZGV0ZWN0cm9uMjsgcHJpbnQoImRldGVjdHJvbjIiLCBkZXRlY3Ryb24yLl9fdmVyc2lvbl9fKQpFT0YKZWNobyAiRG9uZS4gQWN0aXZhdGUgd2l0aDogY29uZGEgYWN0aXZhdGUgJEVOVl9OQU1FICAgKHJlcG86ICRXT1JLRElSL3VuaWxtL2xheW91dGxtdjMpIgoK","examples/run_funsd_cord.py":"..."}''')
base_dir = '/content/unilm/layoutlmv3'
for rel_path, b64_data in FILES_B64.items():
    full_path = os.path.join(base_dir, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'wb') as f:
        f.write(base64.b64decode(b64_data))
print('Synced', len(FILES_B64), 'files to', base_dir)

# 3. Bootstrap Conda Python 3.7 + PyTorch 1.10+cu111 + LayoutLMv3 environment
env = os.environ.copy()
env['WORKDIR'] = '/content'
env['REPO_URL'] = 'https://github.com/pn-minh/unilm.git'
subprocess.run(['bash', '/content/unilm/layoutlmv3/scripts/setup_env.sh'], env=env, check=True)

## 3. Smoke-Test Auxiliary Boundary Loss Implementation
Verifies that LayoutLMv3ForTokenClassification initializes oundary_classifier, extracts B-* entity-start targets (1 for B-*,   for I-*/O, -100 for special/visual/padding tokens), and computes the weighted auxiliary boundary loss.

In [ ]:
%%bash
set -euo pipefail
set +u
if [ -f "$HOME/miniconda3/etc/profile.d/conda.sh" ]; then
  source "$HOME/miniconda3/etc/profile.d/conda.sh"
else
  source "$(conda info --base)/etc/profile.d/conda.sh"
fi
conda activate layoutlmv3
set -u
cd /content/unilm/layoutlmv3

python - <<'EOF'
import torch
from layoutlmft.models.layoutlmv3 import LayoutLMv3Config, LayoutLMv3ForTokenClassification

# Test with FUNSD-like BIO scheme (7 labels, B-* at [1, 3, 5])
# Note: 4 * coordinate_size + 2 * shape_size must equal hidden_size (4*16 + 2*16 = 96)
config = LayoutLMv3Config(
    vocab_size=100,
    hidden_size=96,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=128,
    max_position_embeddings=514,
    max_2d_position_embeddings=1024,
    coordinate_size=16,
    shape_size=16,
    visual_embed=False,
    num_labels=7,
    use_boundary_loss=True,
    boundary_loss_weight=0.2,
    boundary_pos_weight=5.0,
    b_label_ids=[1, 3, 5],
)
model = LayoutLMv3ForTokenClassification(config)
input_ids = torch.randint(0, 100, (2, 16))
bbox = torch.zeros((2, 16, 4), dtype=torch.long)
attention_mask = torch.ones((2, 16), dtype=torch.long)
labels = torch.tensor([
    [-100, 0, 1, 2, 3, 4, 0, 5, 6, 0, -100, -100, -100, -100, -100, -100],
    [-100, 1, 2, 0, 3, 0, 5, 6, 6, 0, -100, -100, -100, -100, -100, -100],
], dtype=torch.long)

out = model(input_ids=input_ids, bbox=bbox, attention_mask=attention_mask, labels=labels)
out.loss.backward()
print('Total loss:', out.loss.item())
print('Main NER loss:', model._last_ner_loss.item())
print('Boundary loss:', model._last_boundary_loss.item())
assert model.boundary_classifier.weight.grad is not None
print('Boundary Loss smoke test PASSED!')
EOF

## 4. Train & Evaluate on FUNSD (Base & Large)
Runs both baseline and Boundary Loss variants on **FUNSD** (ase and large) and syncs each completed run directory to Google Drive (if mounted or configured via rclone).

In [ ]:
%%bash
set -euo pipefail
set +u
if [ -f "$HOME/miniconda3/etc/profile.d/conda.sh" ]; then
  source "$HOME/miniconda3/etc/profile.d/conda.sh"
else
  source "$(conda info --base)/etc/profile.d/conda.sh"
fi
conda activate layoutlmv3
set -u
cd /content/unilm/layoutlmv3

if [ -d /content/drive/MyDrive ]; then
  export DRIVE_MOUNT=/content/drive/MyDrive
fi
export DRIVE_DIR=thesis/layoutlmv3
export OUT=/content/runs

# FUNSD Base & Large (Baseline + Boundary Loss)
DATASETS="funsd" SIZES="base large" VARIANTS="baseline boundary" \
  BOUNDARY_LOSS_WEIGHT=0.2 BOUNDARY_POS_WEIGHT=5.0 \
  bash scripts/run_experiments.sh

## 5. Train & Evaluate on CORD (Base & Large)
Runs both baseline and Boundary Loss variants on **CORD** (ase and large) and updates the aggregated summary on Google Drive.

In [ ]:
%%bash
set -euo pipefail
set +u
if [ -f "$HOME/miniconda3/etc/profile.d/conda.sh" ]; then
  source "$HOME/miniconda3/etc/profile.d/conda.sh"
else
  source "$(conda info --base)/etc/profile.d/conda.sh"
fi
conda activate layoutlmv3
set -u
cd /content/unilm/layoutlmv3

if [ -d /content/drive/MyDrive ]; then
  export DRIVE_MOUNT=/content/drive/MyDrive
fi
export DRIVE_DIR=thesis/layoutlmv3
export OUT=/content/runs
export PER_DEVICE=4

# CORD Base & Large (Baseline + Boundary Loss)
DATASETS="cord" SIZES="base large" VARIANTS="baseline boundary" \
  BOUNDARY_LOSS_WEIGHT=0.2 BOUNDARY_POS_WEIGHT=5.0 \
  bash scripts/run_experiments.sh

## 6. Inspect Aggregated Results

In [ ]:
import os, pandas as pd
from IPython.display import display, Markdown

for base in ['/content/drive/MyDrive/thesis/layoutlmv3/summary', '/content/runs/summary']:
    summary_csv = os.path.join(base, 'summary_metrics.csv')
    summary_md = os.path.join(base, 'summary_metrics.md')
    if os.path.exists(summary_md):
        print('Loaded summary from:', base)
        with open(summary_md, 'r', encoding='utf-8') as f:
            display(Markdown(f.read()))
        if os.path.exists(summary_csv):
            display(pd.read_csv(summary_csv))
        break